# MIMIC-IV-CRPA

In [ ]:
from __future__ import annotations

import numpy as np
import pandas as pd
import os
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, roc_curve,
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

import shap
import warnings
warnings.filterwarnings('ignore')

from scipy.stats import t as student_t

import matplotlib.pyplot as plt
import itertools
from itertools import combinations
from tqdm import tqdm


METRIC_NAMES = ["Accuracy", "Precision", "Recall", "F1-score", "AUROC", "AUPRC"]
MODEL_COLORS_ROC = ['#81989B', '#D19246', '#B5AF8B', '#7EA4B6', '#71A682', '#86BC79', '#B8DBB3', '#4A4F7E']
METRIC_COLORS = ['#6AD1A3', '#7FBDDA', '#BBC7BE','#FFD47D', '#FFA288', '#C49892']
MODEL_COLORS_BAR = ['#6AD1A3', '#7FBDDA', '#BBC7BE', '#FFD47D', '#FFA288', '#C49892', '#84ADDC']

BASE_DIR = "../../../datasets/MIMIC-IV-CRPA/"
INPUT_PATH = BASE_DIR + "crpa_merged_master.csv"
SHAP_KERNEL_BG = 60
SHAP_KERNEL_VAL = 120
SHAP_LINEAR_BG = 200
SAVE_DIR = "../../../results/crpa"
TARGET = "mortality_28d_all_cause"
os.makedirs(SAVE_DIR, exist_ok=True)

# prepare

In [2]:
# =========================================================
# MIMIC-IV-CRPA: Final variable list & preprocessors
# =========================================================
target_col = "mortality_28d_all_cause"
MIMIC_BINARY = [
    "gender_male", "diabetes", "hypertension", "heart_disease", "cerebrovascular_disease",
    "malignancy", "ckd", "chronic_liver_disease", "copd", "hiv_aids", "polymicrobial",
    "has_gram_negative_non_pa", "has_gram_positive_non_pa", "has_unknown_non_pa",
    "resp_invasivevent", "resp_supplementaloxygen", "resp_tracheostomy", "rrt_any",
    "carbapenem_exposure", "piptazo_exposure", "ceph_exposure", "fqn_exposure",
    "aminoglycoside_exposure", "other_abx_exposure",
]
MIMIC_CONTINUOUS = [
    "age", "sofa_closest_to_t0_plus_48h", "alt_worst_4d", "aptt_worst_4d", "ast_worst_4d",
    "bicarbonate_worst_4d", "bilirubin_total_worst_4d", "calcium_total_worst_4d",
    "creatinine_worst_4d", "dbp_worst_4d", "hemoglobin_worst_4d", "hr_worst_4d",
    "inr_worst_4d", "lactate_worst_4d", "magnesium_worst_4d", "map_worst_4d",
    "phosphate_worst_4d", "platelets_worst_4d", "potassium_worst_4d", "pt_worst_4d",
    "rbc_worst_4d", "rdw_worst_4d", "rr_worst_4d", "sbp_worst_4d", "sodium_worst_4d",
    "spo2_worst_4d", "temp_worst_4d", "wbc_worst_4d",
]
all_features = MIMIC_BINARY + MIMIC_CONTINUOUS

# -----------------------------
# feature selection (MIMIC-IV-CRPA)
# -----------------------------
def feature_selection(df, target):
    # MIMIC-IV-CRPA: 28d all-cause mortality
    mimic_all = MIMIC_BINARY + MIMIC_CONTINUOUS
    feature_cols = [c for c in mimic_all if c in df.columns]

    medication_features = [
        "carbapenem_exposure", "piptazo_exposure", "ceph_exposure",
        "fqn_exposure", "aminoglycoside_exposure", "other_abx_exposure",
    ]
    medication_features = [c for c in medication_features if c in df.columns]

    group_defs = {"Comorbidity": ["diabetes", "hypertension", "heart_disease", "cerebrovascular_disease",
        "malignancy", "ckd", "chronic_liver_disease", "copd", "hiv_aids"]}
    include_med = True
    return feature_cols, df.copy(), medication_features, group_defs, include_med

In [3]:
# -----------------------------
# Models
# -----------------------------
def get_models(spw):
    models={
        'XGBoost': XGBClassifier(scale_pos_weight=spw, use_label_encoder=False, objective='binary:logistic', eval_metric='logloss', verbosity=0, random_state=42, n_jobs=1),
        'LightGBM': LGBMClassifier(class_weight='balanced', verbosity=-1, random_state=42, n_jobs=1),
        'CatBoost': CatBoostClassifier(auto_class_weights='Balanced', verbose=0, loss_function='Logloss', allow_writing_files=False, random_state=42, thread_count=1),
        'RandomForest': RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=1),
        'LogisticRegression': LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
        # 'SVM': SVC(class_weight='balanced',random_state=42, probability=True),
        # 'KNN': KNeighborsClassifier(weights='distance', n_jobs=1),
    }
    return models

In [ ]:
# -----------------------------
# Pipeline utilities
# -----------------------------
TREE_MODELS = {"XGBoost", "LightGBM", "CatBoost", "RandomForest"}
def is_tree(model_name): 
    return model_name in TREE_MODELS


def create_pipeline(model, model_name, columns=None, use_smote=False, smote_ratio=0.8):
    # MIMIC: preprocessor for given columns (or full median_preprocessor if columns is None)
    binary_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent"))])
    continuous_transformer_median = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    bin_cols = [c for c in columns if c in MIMIC_BINARY]
    cont_cols = [c for c in columns if c in MIMIC_CONTINUOUS]
    preprocessor = ColumnTransformer(
        transformers=[
            ("bin", binary_transformer, bin_cols),
            ("cont", continuous_transformer_median, cont_cols),
        ],
        remainder="drop"
    )
    
    steps = [("preprocessor", preprocessor)]
    if use_smote:
        steps.append(("smote", SMOTE(random_state=42, sampling_strategy=smote_ratio)))
    steps.append(("clf", clone(model)))
    if use_smote:
        return ImbPipeline(steps)
    return Pipeline(steps)


def get_preprocess_transformer(pipe: Pipeline):
    if "clf" not in pipe.named_steps:
        raise ValueError("Pipeline must have a 'clf' step.")
    return pipe[:-1]

In [ ]:
# -----------------------------
# Metrics / CV report
# -----------------------------
def evaluate_model(y_true, y_pred, y_proba):
    return {
        "Accuracy": float(accuracy_score(y_true, y_pred)),
        "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "F1-score": float(f1_score(y_true, y_pred, zero_division=0)),
        "AUROC": float(roc_auc_score(y_true, y_proba)),
        "AUPRC": float(average_precision_score(y_true, y_proba)),
    }

def cv_metrics(model, model_name, X, y, cv=5, use_smote=False, threshold=0.5, random_state=42, return_ci=False, n_fpr=200):

    kfold = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    per_fold = {k: [] for k in METRIC_NAMES}
    tprs, aucs = [], []
    mean_fpr = np.linspace(0, 1, n_fpr) if return_ci else None

    for tr, va in kfold.split(X, y):
        X_tr, X_va = X.iloc[tr], X.iloc[va]
        y_tr, y_va = y.iloc[tr].values, y.iloc[va].values

        cols = X.columns.tolist()
        pipe = create_pipeline(model if return_ci else clone(model), model_name, columns=cols, use_smote=use_smote)
        pipe.fit(X_tr, y_tr)

        proba = pipe.predict_proba(X_va)[:, 1]
        pred = (proba >= threshold).astype(int)

        metrics = evaluate_model(y_va, pred, proba)
        for k in METRIC_NAMES:
            per_fold[k].append(float(metrics[k]))

        if return_ci:
            fpr, tpr, _ = roc_curve(y_va, proba)
            auc = roc_auc_score(y_va, proba)
            tpr_i = np.interp(mean_fpr, fpr, tpr)
            tpr_i[0] = 0.0
            tprs.append(tpr_i)
            aucs.append(float(auc))

    # calculate 6 metrics mean
    metrics_mean = {k: float(np.mean(per_fold[k])) for k in METRIC_NAMES}

    if not return_ci:
        return metrics_mean

    # calculate CI
    n = cv
    dof = n - 1
    t_val = student_t.ppf(0.975, dof) if n > 1 else 0.0

    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    std_tpr = np.std(tprs, axis=0, ddof=1) if n > 1 else np.zeros_like(mean_tpr)
    tpr_upper = np.clip(mean_tpr + t_val * std_tpr / np.sqrt(n), 0, 1)
    tpr_lower = np.clip(mean_tpr - t_val * std_tpr / np.sqrt(n), 0, 1)

    mean_auc = float(np.mean(aucs))
    std_auc = float(np.std(aucs, ddof=1)) if n > 1 else 0.0
    auc_ci = (mean_auc - t_val * std_auc / np.sqrt(n), mean_auc + t_val * std_auc / np.sqrt(n))

    metrics_ci = {}
    for k in METRIC_NAMES:
        vals = np.array(per_fold[k], dtype=float)
        s = float(np.std(vals, ddof=1)) if n > 1 else 0.0
        m = metrics_mean[k]
        metrics_ci[k] = (m - t_val * s / np.sqrt(n), m + t_val * s / np.sqrt(n))

    return {
        "mean_fpr": mean_fpr,
        "mean_tpr": mean_tpr,
        "tpr_upper": tpr_upper,
        "tpr_lower": tpr_lower,
        "aucs": aucs,
        "mean_auc": mean_auc,
        "std_auc": std_auc,
        "auc_ci": auc_ci,
        "metrics_per_fold": per_fold,
        "metrics_mean": metrics_mean,
        "metrics_ci": metrics_ci,
    }

# SHAP

In [6]:
# -----------------------------
# CV-SHAP for all models
# -----------------------------
def _sample_indices(n, k, seed):
    if k is None or k <= 0 or k >= n:
        return np.arange(n)
    rng = np.random.RandomState(seed)
    return rng.choice(n, size=k, replace=False)

def _ensure_shap_2d(shap_vals):
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]
    shap_vals = np.asarray(shap_vals)
    if shap_vals.ndim == 3 and shap_vals.shape[2] == 2:
        shap_vals = shap_vals[:, :, 1]
    return shap_vals


def cv_shap_importance(models, X, y, cv=5, random_state=42,
    medication_features=None, group_defs=None, exclude_medication_in_ranking=True,
    use_smote=False, verbose=True, return_oof_shap=True,
    kernel_background_samples=SHAP_KERNEL_BG, kernel_val_samples=SHAP_KERNEL_VAL, linear_background_samples=SHAP_LINEAR_BG):
    """
    For each model:
      - 5-fold CV:
          fit pipeline on train_fold
          compute SHAP on val_fold (OOF) in TRANSFORMED space (after impute/scale)
      - aggregate mean(|SHAP|) across folds
      - exclude medication_features from ranking (if set)
      - collapse two groups (Comorbidity, Immunosuppression) by summing SHAP importances
      - return feature_importance, feature_importance_grouped
    """
    assert isinstance(X, pd.DataFrame), "X must be a pandas DataFrame"
    y = pd.Series(y).astype(int)

    medication_features = medication_features or []
    group_defs = group_defs or {"Comorbidity": [], "Immunosuppression": []}

    # map feature -> group (only two groups)
    feat_to_group = {}
    for g, cols in group_defs.items():
        for c in cols:
            if c in X.columns:
                feat_to_group[c] = g

    kfold = StratifiedKFold(n_splits=cv, shuffle=True, random_state=random_state)
    results = {}

    for model_name, model in models.items():
        if verbose:
            print(f"\n===== CV-SHAP (pipeline): {model_name} =====")

        fold_imps = []
        list_shap = []
        list_Xt = []

        for fold, (train, val) in enumerate(kfold.split(X, y), start=1):
            X_train, X_val = X.iloc[train], X.iloc[val]
            y_train = y.iloc[train].values

            pipe = create_pipeline(model, model_name, columns=X.columns.tolist(), use_smote=use_smote)
            pipe.fit(X_train, y_train)

            preprocess = get_preprocess_transformer(pipe)
            clf = pipe.named_steps["clf"]

            # Transform to model feature space (numeric array)
            Xt_train = preprocess.transform(X_train)
            Xt_val = preprocess.transform(X_val)

            # choose explainer by model type
            if is_tree(model_name):
                explainer = shap.TreeExplainer(clf)
                shap_vals = explainer.shap_values(Xt_val)
                shap_vals = _ensure_shap_2d(shap_vals)
                Xt_used = Xt_val

            elif model_name == "LogisticRegression":
                # background in transformed space
                idx_bg = _sample_indices(Xt_train.shape[0], linear_background_samples, seed=random_state + fold)
                Xt_bg = Xt_train[idx_bg]
                explainer = shap.LinearExplainer(clf, Xt_bg, feature_perturbation="interventional")
                shap_vals = explainer.shap_values(Xt_val)
                shap_vals = _ensure_shap_2d(shap_vals)
                Xt_used = Xt_val

            else:
                # KernelExplainer for SVM/KNN (slow): subsample background and validation
                idx_bg = _sample_indices(Xt_train.shape[0], kernel_background_samples, seed=random_state + fold)
                Xt_bg = Xt_train[idx_bg]

                idx_va = _sample_indices(Xt_val.shape[0], kernel_val_samples, seed=random_state + 1000 + fold)
                Xt_va_use = Xt_val[idx_va]

                # function expects transformed features
                def f(z):
                    z = np.asarray(z)
                    if hasattr(clf, "predict_proba"):
                        return clf.predict_proba(z)[:, 1]
                    if hasattr(clf, "decision_function"):
                        s = clf.decision_function(z)
                        return 1.0 / (1.0 + np.exp(-s))
                    raise ValueError(f"{clf.__class__.__name__} lacks predict_proba and decision_function.")

                explainer = shap.KernelExplainer(f, Xt_bg)
                shap_vals = explainer.shap_values(Xt_va_use, nsamples="auto")
                shap_vals = _ensure_shap_2d(shap_vals)
                Xt_used = Xt_va_use

            # align with original feature columns (one-hot already, preprocess doesn't change feature count)
            if shap_vals.ndim != 2 or shap_vals.shape[1] != X.shape[1]:
                raise ValueError(
                    f"{model_name} fold {fold}: SHAP shape {shap_vals.shape} != (n, {X.shape[1]}). "
                    "Make sure X is already one-hot numeric and pipeline doesn't change feature count."
                )
            if return_oof_shap:
                list_shap.append(shap_vals)
                list_Xt.append(Xt_used)
            mean_abs = np.abs(shap_vals).mean(axis=0)
            fold_imps.append({feat: float(mean_abs[i]) for i, feat in enumerate(X.columns)})

            if verbose:
                print(f"  fold {fold}: done (val_n={len(X_val)})")

        # average across folds
        avg_imp = {feat: float(np.mean([d.get(feat, 0.0) for d in fold_imps])) for feat in X.columns}
        raw_df = (
            pd.DataFrame({"feature": list(avg_imp.keys()), "mean_abs_shap": list(avg_imp.values())})
            .sort_values("mean_abs_shap", ascending=False)
            .reset_index(drop=True)
        )

        if exclude_medication_in_ranking and medication_features:
            raw_rank_df = raw_df[~raw_df["feature"].isin(medication_features)].reset_index(drop=True)
        else:
            raw_rank_df = raw_df.copy()

        # group aggregation (only two groups collapsed)
        group_scores = {}
        
        for _, row in raw_rank_df.iterrows():
            feat = str(row["feature"])
            val = float(row["mean_abs_shap"])
            g = feat_to_group.get(feat, feat)
            group_scores[g] = group_scores.get(g, 0.0) + val

        grouped_df = (
            pd.DataFrame({"group": list(group_scores.keys()), "mean_abs_shap": list(group_scores.values())})
            .sort_values("mean_abs_shap", ascending=False)
            .reset_index(drop=True)
        )

        results[model_name] = {
            "feature_importance": raw_df,
            "feature_importance_grouped": grouped_df,
        }
        if return_oof_shap and list_shap:
            results[model_name]["shap_oof"] = np.vstack(list_shap)
            results[model_name]["Xt_oof"] = np.vstack(list_Xt)
            results[model_name]["feature_names"] = X.columns.tolist()

    return results

In [7]:
def draw_shap_bar_beeswarm_rose(shap_results, model_name, group_defs, top_k=15, figsize=(16, 10), cmap_name="PuOr_r", random_state=42, subplot_label="(a)", show=True):
    importance_df = shap_results[model_name]["feature_importance"]
    shap_matrix = shap_results[model_name]["shap_oof"]
    X_for_plot = shap_results[model_name]["Xt_oof"]
    feature_names = shap_results[model_name]["feature_names"]

    comorb_cols = group_defs.get("Comorbidity", [])
    immuno_cols = group_defs.get("Immunosuppression", [])
    feats_in_df = set(importance_df["feature"].values)
    comorb_in = [f for f in comorb_cols if f in feats_in_df]
    immuno_in = [f for f in immuno_cols if f in feats_in_df]
    other_features = [
        f for f in importance_df["feature"].values
        if f not in comorb_cols and f not in immuno_cols
    ]

    def _sum_imp(feats):
        return float(importance_df.loc[importance_df["feature"].isin(feats), "mean_abs_shap"].sum())

    imp_comorb = _sum_imp(comorb_in) if comorb_in else 0.0
    imp_immuno = _sum_imp(immuno_in) if immuno_in else 0.0
    imp_others = importance_df.set_index("feature").loc[other_features, "mean_abs_shap"].to_dict()

    display_items = []
    if comorb_cols:
        display_items.append(("Comorbidity", imp_comorb))
    if immuno_cols:
        display_items.append(("Immunosuppression", imp_immuno))
    for f in other_features:
        display_items.append((f, imp_others.get(f, 0.0)))
    display_items.sort(key=lambda x: x[1], reverse=True)
    display_items = display_items[:top_k]
    sorted_display_names = np.array([x[0] for x in display_items])
    sorted_importance = np.array([x[1] for x in display_items])
    n_features = len(sorted_display_names)
    n_samples = shap_matrix.shape[0]

    name_to_idx = {f: i for i, f in enumerate(feature_names)}
    shap_by_feature = np.zeros((n_samples, n_features))
    feature_values = np.zeros((n_samples, n_features))
    X_min = X_for_plot.min(axis=0)
    X_max = X_for_plot.max(axis=0)
    X_range = np.where(X_max > X_min, X_max - X_min, 1.0)
    X_norm = (X_for_plot - X_min) / X_range

    for j, name in enumerate(sorted_display_names):
        if name == "Comorbidity":
            cols = [name_to_idx[f] for f in comorb_in if f in name_to_idx]
            if cols:
                shap_by_feature[:, j] = shap_matrix[:, cols].sum(axis=1)
                feature_values[:, j] = X_norm[:, cols].mean(axis=1)
            else:
                feature_values[:, j] = 0.5
        elif name == "Immunosuppression":
            cols = [name_to_idx[f] for f in immuno_in if f in name_to_idx]
            if cols:
                shap_by_feature[:, j] = shap_matrix[:, cols].sum(axis=1)
                feature_values[:, j] = X_norm[:, cols].mean(axis=1)
            else:
                feature_values[:, j] = 0.5
        else:
            if name in name_to_idx:
                idx = name_to_idx[name]
                shap_by_feature[:, j] = shap_matrix[:, idx]
                feature_values[:, j] = X_norm[:, idx]

    vmax_imp = float(sorted_importance.max()) if n_features else 1.0
    cmap = plt.get_cmap(cmap_name)
    norm_imp = plt.Normalize(vmin=0, vmax=vmax_imp)
    norm_feat = plt.Normalize(vmin=0, vmax=1)

    fig = plt.figure(figsize=figsize, constrained_layout=False)
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.2], wspace=0.0)
    ax_bar = fig.add_subplot(gs[0])
    ax_shap = fig.add_subplot(gs[1])

    y_pos = np.arange(n_features)
    ax_bar.barh(y_pos, sorted_importance, height=0.65,
                color=cmap(norm_imp(sorted_importance)), edgecolor="none")
    ax_bar.set_xlim(vmax_imp, 0)
    for i in range(n_features):
        ax_bar.text(0, y_pos[i], sorted_display_names[i], ha='right', va='center', fontsize=11, color='black')
    ax_bar.set_ylim(-0.5, n_features - 0.5)
    ax_bar.invert_yaxis()
    ax_bar.yaxis.tick_right()
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels([])
    ax_bar.tick_params(axis="y", length=0, pad=10)
    ax_bar.set_xlabel("Mean |SHAP| (contribution)", fontsize=11)
    ax_bar.spines["left"].set_visible(False)
    ax_bar.spines["top"].set_visible(False)
    ax_bar.spines["right"].set_visible(False)
    ax_bar.spines["bottom"].set_visible(True)

    cax1 = ax_bar.inset_axes([-0.1, 0, 0.02, 1])
    cb1 = plt.colorbar(plt.cm.ScalarMappable(norm=norm_imp, cmap=cmap), cax=cax1)
    cb1.set_ticks([])
    cb1.outline.set_visible(False)
    cax1.text(0.5, 1.02, "High", ha="center", va="bottom", transform=cax1.transAxes, fontsize=10)
    cax1.text(0.5, -0.02, "Low", ha="center", va="top", transform=cax1.transAxes, fontsize=10)

    np.random.seed(random_state)
    for i in range(n_features):
        y_jitter = np.random.normal(loc=i, scale=0.08, size=n_samples)
        ax_shap.scatter(shap_by_feature[:, i], y_jitter,
                        c=feature_values[:, i], cmap=cmap, s=10,
                        alpha=0.8, edgecolors="none", norm=norm_feat)
    ax_shap.axvline(0, color="grey", linewidth=0.5, alpha=0.5)
    ax_shap.set_ylim(-0.5, n_features - 0.5)
    ax_shap.invert_yaxis()
    ax_shap.set_yticks([])
    ax_shap.set_xlabel("SHAP Value (impact on model output)", fontsize=11)
    ax_shap.spines["top"].set_visible(False)
    ax_shap.spines["right"].set_visible(False)
    ax_shap.spines["left"].set_visible(False)

    cax2 = ax_shap.inset_axes([1.05, 0, 0.02, 1])
    cb2 = plt.colorbar(plt.cm.ScalarMappable(norm=norm_feat, cmap=cmap), cax=cax2)
    cb2.set_ticks([])
    cb2.set_label("Feature Value", rotation=270, labelpad=15)
    cb2.outline.set_visible(False)
    cax2.text(0.5, 1.02, "High", ha="center", va="bottom", transform=cax2.transAxes, fontsize=10)
    cax2.text(0.5, -0.02, "Low", ha="center", va="top", transform=cax2.transAxes, fontsize=10)

    inset_ax_rect = [0.05, 0.12, 0.38, 0.38]
    ax_rose = fig.add_axes(inset_ax_rect, projection='polar')
    ax_rose.patch.set_alpha(0)
    num_vars = n_features
    widths = (sorted_importance / np.maximum(sorted_importance.sum(), 1e-12)) * 2 * np.pi
    widths = np.asarray(widths)
    base_length, fixed_increment, colored_ring_width = 3.0, 0.5, 2.0
    total_lengths = np.array([base_length + i * fixed_increment for i in range(num_vars)])
    inner_heights = np.maximum(0, total_lengths - colored_ring_width)
    inner_colors = ['#EAEAEA', '#FFFFFF'] * (num_vars // 2 + 1)
    inner_colors = inner_colors[:num_vars]
    one_oclock_offset = np.pi / 21
    thetas = np.asarray(np.cumsum([0] + widths[:-1].tolist()) - one_oclock_offset)

    ax_rose.bar(x=thetas, height=inner_heights, width=widths, color=inner_colors,
                align='edge', edgecolor='white', linewidth=1.5)
    ax_rose.bar(x=thetas, height=[colored_ring_width] * num_vars, width=widths, bottom=inner_heights,
                color=cmap(norm_imp(sorted_importance)), align='edge', edgecolor='white', linewidth=1.5)
    ax_rose.set_yticklabels([])
    ax_rose.set_xticklabels([])
    ax_rose.spines['polar'].set_visible(False)
    ax_rose.grid(False)
    ax_rose.set_theta_zero_location('N')
    ax_rose.set_theta_direction(-1)
    ax_rose.set_ylim(0, max(total_lengths) + 2)

    total_imp = float(np.sum(sorted_importance))
    thresh = float(np.max(sorted_importance) * 0.1) if n_features else 0
    outer_radii = inner_heights + colored_ring_width
    for i in range(num_vars):
        if sorted_importance[i] >= thresh and total_imp > 0:
            pct = sorted_importance[i] / total_imp * 100
            theta_center = thetas[i] + widths[i] / 2
            label_r = outer_radii[i] * 1.2
            ax_rose.text(theta_center, label_r, f"{pct:.1f}%", ha="center", va="center", fontsize=8, rotation=0)

    ax_bar.text(0.0, 1.02, subplot_label, transform=ax_bar.transAxes, fontsize=20, fontweight="bold")
    if show:
        plt.show()

# Feature-set selection

## Stage A

In [8]:
# -----------------------------
# 1) group_defs -> raw columns
# -----------------------------
def build_group_to_cols(feature_cols, group_defs):
    """
    feature_cols: List[str]             raw columns used in training (already one-hot / imputer + scaler)
    return: group_to_cols: Dict[str, List[str]]    eg. {"Comorbidity":[...], "Immunosuppression":[...]}
    """
    # only include columns that exist in feature_cols (avoid KeyError from missing e.g. hiv_aids)
    group_to_cols = {g: [c for c in cols if c in feature_cols] for g, cols in group_defs.items()}
    grouped_cols = set(c for cols in group_to_cols.values() for c in cols)

    for c in feature_cols:
        if c in grouped_cols:
            continue
        group_to_cols.setdefault(c, [c])  # singleton group

    return group_to_cols


def groups_to_feature_cols(groups, group_to_cols, medication_features = None, include_med = False):
    """
    groups -> expanded raw columns (dedup, keep order).
    Optionally prepend medication_features when include_med=True.
    """
    cols = []
    for g in groups:
        cols.extend(group_to_cols.get(g, [g]))

    # de-dup keep order
    cols = list(dict.fromkeys(cols))

    if include_med and medication_features:
        cols = list(dict.fromkeys(medication_features + cols))

    return cols

In [9]:
# -----------------------------
# 2) enumerate group subsets
# -----------------------------
def enumerate_group_subsets(top_groups, max_k=10):
    n = len(top_groups)
    top_groups = top_groups[:max_k]
    for k in range(1, min(max_k, n) + 1):
        for comb in combinations(top_groups, k):
            yield list(comb)

In [ ]:
# -----------------------------
# 3)for each model: search top sets by AUROC(top10 feature->top7 feature-set)
# ----------------------------- 
def search_topsets_per_model(model, model_name, X_train, y_train, top10_groups, group_to_cols, medication_features, include_med, cv=5, use_smote=False, top_sets=7, max_k=10):
    """
    to per model: enumerate all subsets of topk_groups, select top_sets sorted by auroc
    return: List[(groups, score, ncols)]
    """
    results = []
    from math import comb
    n = min(len(top10_groups), max_k)
    actual_total = sum(comb(n, k) for k in range(1, n + 1)) if n > 0 else 0
    for groups in tqdm(enumerate_group_subsets(top10_groups, max_k=max_k), desc=model_name, total=actual_total):        
        cols = groups_to_feature_cols(groups, group_to_cols, medication_features, include_med)
        # Score subset by AUROC
        score = cv_metrics(model, model_name, X=X_train[cols], y=y_train, cv=cv, use_smote=use_smote)["AUROC"]
        results.append((groups, score, len(cols)))

    results.sort(key=lambda x: x[1], reverse=True)
    return results[:top_sets]


In [ ]:
# -----------------------------
# union and dedup candidates
# ----------------------------- 
def union_dedup_candidates(per_model_top_sets):
    seen = set()
    union = []
    for model_name, lst in per_model_top_sets.items():
        for candidates, score, ncols in lst:
            key = tuple(candidates)
            if key not in seen:
                seen.add(key)
                union.append({"candidates": candidates, "seed_score": score, "ncols": ncols})
    return union

In [ ]:
# =========================================================
# 4) get topset per model
# =========================================================
def search_candidates(X_train, y_train, feature_cols, group_defs, models, shap_results, medication_features, include_med, cv=5, use_smote=False, top_sets=7, max_k=10, verbose=True):
    """
    Input:
      - feature_cols, group_defs:   from feature_selection (no need to recalculate)
      - models:                     model dictionary (no need to recalculate)
      - shap_results:               each model's top10_groups (you already have)
      - include_med:                (outcome/survival/duration=True; resistance=False)

    Output:
      - per_model_top_sets: each model's top7 candidate sets (groups-level)
      - group_to_cols: for later expand + full evaluation
    """
    # group -> raw cols
    group_to_cols = build_group_to_cols(feature_cols, group_defs)
    per_model_top_sets = {}

    for model_name, model in models.items():
        top10 = shap_results[model_name]["feature_importance_grouped"]["group"].head(10).tolist()
        per_model_top_sets[model_name] = search_topsets_per_model(model, model_name, X_train, y_train, top10[:max_k], group_to_cols, medication_features, include_med, cv, use_smote, top_sets, max_k)
        if verbose:
            print(f"model: {model_name}, top10: {top10}")
            for i in per_model_top_sets[model_name]:
                print(i)
    return per_model_top_sets, group_to_cols

## Stage B

In [13]:
def evaluate_candidates(
    candidates, models, X_train, y_train, group_to_cols,
    medication_features, include_med,
    cv=5, use_smote=False, random_state=42, threshold=0.5
):
    rows = []
    candidate_details = {}

    for i, cand in enumerate(candidates, start=1):
        set_id = f"cand_{i:02d}"
        groups = cand["candidates"]
        cols = groups_to_feature_cols(groups, group_to_cols, medication_features, include_med)
        X_sub = X_train[cols]

        per_model_metrics = {}
        for model_name, model in models.items():
            per_model_metrics[model_name] = cv_metrics(model=model, model_name=model_name, X=X_sub, y=y_train, cv=cv, use_smote=use_smote, random_state=random_state, threshold=threshold,return_ci=False)

        # final metric: set's mean over models
        set_metric_mean = {}
        for k in METRIC_NAMES:
            set_metric_mean[k] = float(np.mean([per_model_metrics[m][k] for m in per_model_metrics]))

        candidate_details[set_id] = {"groups": groups, "cols": cols, "per_model_metrics": per_model_metrics}

        row = {"set_id": set_id, "groups": "|".join(groups), "n_groups": len(groups), "n_cols": len(cols)}
        for k in METRIC_NAMES:
            row[f"{k}_mean_over_models"] = set_metric_mean[k]
        rows.append(row)

    set_scores_df = pd.DataFrame(rows)
    set_scores_df = set_scores_df.sort_values("AUROC_mean_over_models", ascending=False).reset_index(drop=True)
    return set_scores_df, candidate_details

## stage C

In [ ]:
def select_final_sets(set_scores_df, candidate_details, drop_pct=0.4, min_models_kept=5):
    best_set_per_metric = {}

    model_names = list(next(iter(candidate_details.values()))["per_model_metrics"].keys())
    set_ids = list(candidate_details.keys())
    n_sets = len(set_ids)
    n_keep = max(1, int(n_sets * (1 - drop_pct)))

    for k in METRIC_NAMES:
        kept_by_model = {}
        for m in model_names:
            scores = {sid: float(candidate_details[sid]["per_model_metrics"][m][k]) for sid in set_ids}
            sorted_sets = sorted(set_ids, key=lambda s: scores[s], reverse=True)
            threshold_score = scores[sorted_sets[n_keep - 1]]
            kept_by_model[m] = set(sid for sid in set_ids if scores[sid] >= threshold_score)

        eligible = set(set_ids)
        for m in model_names:
            eligible = eligible & kept_by_model[m]
        eligible = list(eligible)

        if not eligible:
            eligible = [sid for sid in set_ids
                       if sum(1 for m in model_names if sid in kept_by_model[m]) >= min_models_kept]
        col = f"{k}_mean_over_models"
        if eligible:
            sub = set_scores_df[set_scores_df["set_id"].isin(eligible)]
            best_set_id = sub.loc[sub[col].idxmax(), "set_id"]
            best_set_per_metric[k] = best_set_id
        else:
            best_set_id = set_scores_df.loc[set_scores_df[col].idxmax(), "set_id"]
            best_set_per_metric[k] = best_set_id

    return best_set_per_metric

## stage D

In [15]:
def evaluate_final_sets(best_set_per_metric, candidate_details, models, X_train, y_train, cv=5, use_smote=False, random_state=42):
    final_set_eval = {}
    metrics_rows = []

    final_set_ids = list(dict.fromkeys(best_set_per_metric.values()))
    set_id_to_metrics = {}
    for metric, set_id in best_set_per_metric.items():
        set_id_to_metrics.setdefault(set_id, []).append(metric)
    
    for set_id in final_set_ids:
        selected_by = set_id_to_metrics[set_id]
        print(f"set_id: {set_id}, selected_by: {selected_by}")

        groups = candidate_details[set_id]["groups"]
        cols = candidate_details[set_id]["cols"]
        X_sub = X_train[cols]

        final_set_eval[set_id] = {"groups": groups, "cols": cols, "selected_by": selected_by, "models": {}}

        for model_name, model in models.items():
            print("="*30, model_name,"="*30,"\n")

            cv_ci = cv_metrics(model, model_name, X=X_sub, y=y_train, cv=cv, use_smote=use_smote, random_state=random_state,return_ci=True)

            final_set_eval[set_id]["models"][model_name] = {
                "cv_ci": cv_ci,
            }

            row = {
                "set_id": set_id, "model_name": model_name, "selected_by": "|".join(selected_by),
                **{k: cv_ci["metrics_mean"][k] for k in METRIC_NAMES},
                "AUROC_lower": cv_ci["metrics_ci"]["AUROC"][0],
                "AUROC_upper": cv_ci["metrics_ci"]["AUROC"][1],
            }
            for k in METRIC_NAMES:
                print(f"{k}: {row[k]:.4f} | ", end="")
            print("\n")
            metrics_rows.append(row)

    final_set_metrics = pd.DataFrame(metrics_rows)
    return final_set_eval, final_set_metrics

# main

In [17]:
# ===== 1. load data (MIMIC-IV-CRPA) =====
df = pd.read_csv(INPUT_PATH)
missing_features = [c for c in all_features + [target_col] if c not in df.columns]
if missing_features:
    raise ValueError(f"Missing columns in input file: {missing_features}")

# ===== feature_selection: feature_cols, group_defs, medication_features, include_med =====
feature_cols, train_df, medication_features, group_defs, include_med = feature_selection(df, target_col)
X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy().astype(int)

print("X shape:", X_train.shape)
print("feature_cols n:", len(feature_cols))
print("group_defs:", group_defs)
print("medication_features:", medication_features)
print("include_med:", include_med)
print("y distribution:")
print(y_train.value_counts(dropna=False).sort_index())

# ===== 2. models =====
scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
models = get_models(scale_pos_weight)

X shape: (286, 52)
feature_cols n: 52
group_defs: {'Comorbidity': ['diabetes', 'hypertension', 'heart_disease', 'cerebrovascular_disease', 'malignancy', 'ckd', 'chronic_liver_disease', 'copd', 'hiv_aids']}
medication_features: ['carbapenem_exposure', 'piptazo_exposure', 'ceph_exposure', 'fqn_exposure', 'aminoglycoside_exposure', 'other_abx_exposure']
include_med: True
y distribution:
mortality_28d_all_cause
0    206
1     80
Name: count, dtype: int64


### Train with all-features

In [ ]:
import numpy as np

per_model_metrics = {}
for model_name, model in models.items():
    per_model_metrics[model_name] = cv_metrics(model, model_name, X_train, y_train, cv=5, use_smote=False, threshold=0.5, random_state=42, return_ci=True, n_fpr=200)


print("Model:              " + " ".join([f"{m}(mean±std)" for m in METRIC_NAMES]))
for model_name, res in per_model_metrics.items():
    print(f"{model_name:18}", end=" ")
    for m in METRIC_NAMES:
        mean_v = res["metrics_mean"][m]
        std_v  = float(np.std(res["metrics_per_fold"][m], ddof=1))
        print(f" {mean_v:.4f}±{std_v:.4f}", end=" ")
    print()

Model:              Accuracy(mean±std) Precision(mean±std) Recall(mean±std) F1-score(mean±std) AUROC(mean±std) AUPRC(mean±std)
XGBoost             0.6997±0.0698  0.4364±0.1711  0.3125±0.1768  0.3563±0.1807  0.6378±0.1022  0.4680±0.1100 
LightGBM            0.6926±0.1106  0.4906±0.2598  0.3875±0.1355  0.4218±0.1636  0.6341±0.0955  0.4841±0.1415 
CatBoost            0.7311±0.0590  0.5407±0.1553  0.4000±0.1218  0.4520±0.1166  0.6740±0.0951  0.5390±0.1179 
RandomForest        0.7520±0.0416  0.8167±0.2911  0.1875±0.1250  0.2843±0.1597  0.6759±0.0859  0.5236±0.1197 
LogisticRegression  0.6365±0.0746  0.3884±0.0917  0.5250±0.1801  0.4410±0.1183  0.6342±0.1021  0.4601±0.1218 


### SHAP -> feature importance rank -> top10_groups per model

In [20]:
# ===== 4. SHAP ranking (pipeline version) =====
shap_results = cv_shap_importance(
        models=models, X=X_train, y=y_train, cv=5,
        medication_features=medication_features if include_med else [],
        group_defs=group_defs, exclude_medication_in_ranking=True,
        use_smote=False, verbose=False, return_oof_shap=True
    )

In [21]:
for model_name in models:
    top10 = shap_results[model_name]["feature_importance_grouped"]["group"].head(10).tolist()
    print(f"{model_name}: {top10}")

XGBoost: ['aptt_worst_4d', 'pt_worst_4d', 'lactate_worst_4d', 'hr_worst_4d', 'rr_worst_4d', 'creatinine_worst_4d', 'age', 'Comorbidity', 'rdw_worst_4d', 'bicarbonate_worst_4d']
LightGBM: ['aptt_worst_4d', 'rr_worst_4d', 'pt_worst_4d', 'hr_worst_4d', 'lactate_worst_4d', 'age', 'creatinine_worst_4d', 'bicarbonate_worst_4d', 'sodium_worst_4d', 'Comorbidity']
CatBoost: ['aptt_worst_4d', 'pt_worst_4d', 'rr_worst_4d', 'lactate_worst_4d', 'hr_worst_4d', 'inr_worst_4d', 'age', 'rdw_worst_4d', 'Comorbidity', 'hemoglobin_worst_4d']
RandomForest: ['aptt_worst_4d', 'inr_worst_4d', 'pt_worst_4d', 'rdw_worst_4d', 'lactate_worst_4d', 'Comorbidity', 'rr_worst_4d', 'age', 'hemoglobin_worst_4d', 'sodium_worst_4d']
LogisticRegression: ['Comorbidity', 'bicarbonate_worst_4d', 'hemoglobin_worst_4d', 'age', 'aptt_worst_4d', 'ast_worst_4d', 'resp_invasivevent', 'creatinine_worst_4d', 'hr_worst_4d', 'has_unknown_non_pa']


In [ ]:
shap_dir=SAVE_DIR+'/shap_csv'
os.makedirs(shap_dir, exist_ok=True)

all_df = pd.concat([res["feature_importance"].assign(model=mn) for mn, res in shap_results.items()], ignore_index=True)
all_df.to_csv(shap_dir + "/importance_with_medication_no_group-nogridsearch.csv", index=False)

all_df = pd.concat([
        res["feature_importance"][~res["feature_importance"]["feature"].isin(medication_features)].assign(model=mn)
        for mn, res in shap_results.items()
    ], ignore_index=True)
all_df.to_csv(shap_dir + "/importance_no_medication_no_group-nogridsearch.csv", index=False)

all_df = pd.concat([res["feature_importance_grouped"].assign(model=mn) for mn, res in shap_results.items()], ignore_index=True)
all_df.to_csv(shap_dir + "/importance_no_medication_with_group-nogridsearch.csv", index=False)

### Get top5 feature-sets per model

In [24]:
per_model_top_sets, group_to_cols = search_candidates(
    X_train, y_train, feature_cols, group_defs, models,
    shap_results, medication_features, include_med, 
    cv=5, use_smote=False, top_sets=5, max_k=10, verbose=True
)

XGBoost:   0%|          | 0/1023 [00:00<?, ?it/s]

XGBoost: 100%|██████████| 1023/1023 [05:12<00:00,  3.27it/s]


model: XGBoost, top10: ['aptt_worst_4d', 'pt_worst_4d', 'lactate_worst_4d', 'hr_worst_4d', 'rr_worst_4d', 'creatinine_worst_4d', 'age', 'Comorbidity', 'rdw_worst_4d', 'bicarbonate_worst_4d']
(['pt_worst_4d', 'rr_worst_4d', 'creatinine_worst_4d', 'Comorbidity', 'rdw_worst_4d', 'bicarbonate_worst_4d'], 0.7428644018583043, 20)
(['pt_worst_4d', 'lactate_worst_4d', 'rr_worst_4d', 'creatinine_worst_4d', 'age', 'Comorbidity', 'rdw_worst_4d'], 0.7428353658536585, 21)
(['pt_worst_4d', 'rr_worst_4d', 'creatinine_worst_4d', 'rdw_worst_4d', 'bicarbonate_worst_4d'], 0.7388356562137052, 11)
(['pt_worst_4d', 'lactate_worst_4d', 'rr_worst_4d', 'creatinine_worst_4d', 'age', 'rdw_worst_4d', 'bicarbonate_worst_4d'], 0.7345891405342625, 13)
(['pt_worst_4d', 'rr_worst_4d', 'creatinine_worst_4d', 'age', 'Comorbidity', 'rdw_worst_4d', 'bicarbonate_worst_4d'], 0.7308507549361207, 21)


LightGBM: 100%|██████████| 1023/1023 [03:10<00:00,  5.36it/s]


model: LightGBM, top10: ['aptt_worst_4d', 'rr_worst_4d', 'pt_worst_4d', 'hr_worst_4d', 'lactate_worst_4d', 'age', 'creatinine_worst_4d', 'bicarbonate_worst_4d', 'sodium_worst_4d', 'Comorbidity']
(['aptt_worst_4d', 'rr_worst_4d', 'pt_worst_4d', 'hr_worst_4d', 'lactate_worst_4d', 'age', 'creatinine_worst_4d', 'bicarbonate_worst_4d'], 0.7219076655052264, 14)
(['rr_worst_4d', 'pt_worst_4d', 'lactate_worst_4d', 'age', 'creatinine_worst_4d', 'bicarbonate_worst_4d'], 0.720397793263647, 12)
(['aptt_worst_4d', 'rr_worst_4d', 'pt_worst_4d', 'hr_worst_4d', 'lactate_worst_4d', 'age', 'creatinine_worst_4d', 'bicarbonate_worst_4d', 'sodium_worst_4d'], 0.7203469802555168, 15)
(['pt_worst_4d', 'hr_worst_4d', 'lactate_worst_4d', 'age', 'creatinine_worst_4d', 'bicarbonate_worst_4d', 'sodium_worst_4d'], 0.7189024390243903, 13)
(['rr_worst_4d', 'pt_worst_4d', 'hr_worst_4d', 'lactate_worst_4d', 'age', 'creatinine_worst_4d', 'bicarbonate_worst_4d', 'sodium_worst_4d'], 0.7174216027874565, 14)


...

model: CatBoost, top10: ['aptt_worst_4d', 'pt_worst_4d', 'rr_worst_4d', 'lactate_worst_4d', 'hr_worst_4d', 'inr_worst_4d', 'age', 'rdw_worst_4d', 'Comorbidity', 'hemoglobin_worst_4d']
(['aptt_worst_4d', 'rr_worst_4d', 'lactate_worst_4d', 'hr_worst_4d', 'inr_worst_4d', 'age', 'rdw_worst_4d'], 0.7121806039488967, 13)
(['pt_worst_4d', 'rr_worst_4d', 'lactate_worst_4d', 'age', 'rdw_worst_4d'], 0.7119773519163763, 11)
(['aptt_worst_4d', 'pt_worst_4d', 'lactate_worst_4d', 'hr_worst_4d', 'age', 'Comorbidity'], 0.7039126016260162, 20)
(['aptt_worst_4d', 'rr_worst_4d', 'lactate_worst_4d', 'rdw_worst_4d'], 0.7037020905923346, 10)
(['rr_worst_4d', 'lactate_worst_4d', 'inr_worst_4d', 'age', 'rdw_worst_4d'], 0.703513356562137, 11)


RandomForest: 100%|██████████| 1023/1023 [16:49<00:00,  1.01it/s]


model: RandomForest, top10: ['aptt_worst_4d', 'inr_worst_4d', 'pt_worst_4d', 'rdw_worst_4d', 'lactate_worst_4d', 'Comorbidity', 'rr_worst_4d', 'age', 'hemoglobin_worst_4d', 'sodium_worst_4d']
(['pt_worst_4d', 'rdw_worst_4d', 'lactate_worst_4d', 'age', 'sodium_worst_4d'], 0.7442109465737514, 11)
(['pt_worst_4d', 'rdw_worst_4d', 'lactate_worst_4d', 'rr_worst_4d', 'age', 'hemoglobin_worst_4d', 'sodium_worst_4d'], 0.7421130952380952, 13)
(['inr_worst_4d', 'pt_worst_4d', 'rdw_worst_4d', 'lactate_worst_4d', 'age', 'hemoglobin_worst_4d', 'sodium_worst_4d'], 0.7354420731707317, 13)
(['aptt_worst_4d', 'pt_worst_4d', 'rdw_worst_4d', 'lactate_worst_4d', 'rr_worst_4d', 'age', 'hemoglobin_worst_4d', 'sodium_worst_4d'], 0.7347052845528456, 14)
(['inr_worst_4d', 'rdw_worst_4d', 'lactate_worst_4d', 'rr_worst_4d', 'age', 'hemoglobin_worst_4d', 'sodium_worst_4d'], 0.7340773809523811, 13)


LogisticRegression: 100%|██████████| 1023/1023 [01:51<00:00,  9.14it/s]

model: LogisticRegression, top10: ['Comorbidity', 'bicarbonate_worst_4d', 'hemoglobin_worst_4d', 'age', 'aptt_worst_4d', 'ast_worst_4d', 'resp_invasivevent', 'creatinine_worst_4d', 'hr_worst_4d', 'has_unknown_non_pa']
(['bicarbonate_worst_4d', 'hemoglobin_worst_4d', 'age', 'aptt_worst_4d', 'ast_worst_4d', 'resp_invasivevent', 'creatinine_worst_4d', 'hr_worst_4d', 'has_unknown_non_pa'], 0.7521341463414635, 15)
(['bicarbonate_worst_4d', 'hemoglobin_worst_4d', 'age', 'aptt_worst_4d', 'ast_worst_4d', 'resp_invasivevent', 'hr_worst_4d', 'has_unknown_non_pa'], 0.7477932636469221, 14)
(['bicarbonate_worst_4d', 'age', 'aptt_worst_4d', 'ast_worst_4d', 'resp_invasivevent', 'creatinine_worst_4d', 'hr_worst_4d', 'has_unknown_non_pa'], 0.7468205574912891, 14)
(['bicarbonate_worst_4d', 'hemoglobin_worst_4d', 'age', 'aptt_worst_4d', 'ast_worst_4d', 'resp_invasivevent', 'creatinine_worst_4d', 'hr_worst_4d'], 0.7445267131242741, 14)
(['bicarbonate_worst_4d', 'age', 'aptt_worst_4d', 'ast_worst_4d', 'res

### Evaluate candidates

In [25]:
candidates = union_dedup_candidates(per_model_top_sets)
set_scores_df, candidate_details = evaluate_candidates(candidates, models, X_train, y_train, group_to_cols, medication_features, include_med, cv=5, use_smote=False)

In [26]:
feature_selection_dir = SAVE_DIR +'/feature_selection'
os.makedirs(feature_selection_dir, exist_ok=True)
set_scores_df.to_csv(feature_selection_dir + "/set_scores_stageB-nogridsearch.csv", index=False, encoding="utf-8-sig")

In [27]:
rows = []
for set_id, d in candidate_details.items():
    groups_str = "|".join(d["groups"])
    for model_name, m in d["per_model_metrics"].items():
        row = {"set_id": set_id, "groups": groups_str, "model": model_name, **m}
        rows.append(row)
pd.DataFrame(rows).to_csv(feature_selection_dir + "/candidate_details-nogridsearch.csv", index=False, encoding="utf-8-sig")
print(f"Saved set_scores and {len(candidate_details)} candidate details to {feature_selection_dir}")

Saved set_scores and 25 candidate details to ../../../results/crpa0411/feature_selection


### Select final feature-set by metric

In [ ]:
best_set_per_metric = select_final_sets(set_scores_df, candidate_details, drop_pct=0.4, min_models_kept=4)
print("best_set_per_metric:", best_set_per_metric)
final_set_ids = list(dict.fromkeys(best_set_per_metric.values()))
print("final_set_ids:", final_set_ids)

best_set_per_metric: {'Accuracy': 'cand_25', 'Precision': 'cand_04', 'Recall': 'cand_25', 'F1-score': 'cand_04', 'AUROC': 'cand_04', 'AUPRC': 'cand_25'}
final_set_ids: ['cand_25', 'cand_04']


### 5-fold on final feature-set

In [30]:
final_set_eval, final_set_metrics = evaluate_final_sets(best_set_per_metric, candidate_details, models, X_train, y_train, cv=5, use_smote=False)
final_set_metrics.to_csv(SAVE_DIR + "/final_set_metrics-nogridsearch.csv", index=False)

set_id: cand_25, selected_by: ['Accuracy', 'Recall', 'AUPRC']
============================== XGBoost ============================== 



Accuracy: 0.7169 | Precision: 0.4978 | Recall: 0.4500 | F1-score: 0.4634 | AUROC: 0.7001 | AUPRC: 0.5586 | 

============================== LightGBM ============================== 

Accuracy: 0.6994 | Precision: 0.4651 | Recall: 0.4500 | F1-score: 0.4474 | AUROC: 0.6744 | AUPRC: 0.5315 | 

============================== CatBoost ============================== 

Accuracy: 0.7206 | Precision: 0.4972 | Recall: 0.4875 | F1-score: 0.4901 | AUROC: 0.6912 | AUPRC: 0.5368 | 

============================== RandomForest ============================== 

Accuracy: 0.7414 | Precision: 0.5895 | Recall: 0.2125 | F1-score: 0.3112 | AUROC: 0.7022 | AUPRC: 0.5321 | 

============================== LogisticRegression ============================== 

Accuracy: 0.6995 | Precision: 0.4711 | Recall: 0.6500 | F1-score: 0.5445 | AUROC: 0.7435 | AUPRC: 0.5695 | 

set_id: cand_04, selected_by: ['Precision', 'F1-score', 'AUROC']
============================== XGBoost ============================== 

Accuracy: 0.